In [ ]:
import mlflow
import mlflow.sklearn

import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns

from mlflow.models import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    ConfusionMatrixDisplay
)

from imblearn.metrics import geometric_mean_score


In [ ]:
def load_data(filepath):
    data = pd.read_csv(filepath)
    return data

def explore_data(data):
    head = data.head()
    columns = data.columns
    data_types = data.dtypes
    return head, columns, data_types


In [ ]:
def clean_data(data):
    data = data.drop(["user_id", "next_day_fatigue_score"], axis=1)
    data["blue_light_filter_active"] = pd.Categorical(
        data["blue_light_filter_active"],
        categories=[0, 1]
    )
    data["sleep_debt_category"] = pd.Categorical(
        data["sleep_debt_category"],
        categories=[
            "Optimal Recovery",
            "Mild Deficit",
            "Moderate Debt",
            "Severe Sleep Debt"
        ],
        ordered=True
    )
    return data

In [ ]:
def explore_variables(data):
    num_cols = data.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = data.select_dtypes(include=["str", "category"]).columns.tolist()
    num_stats = data[num_cols].describe().T
    num_stats["Skew"] = data[num_cols].skew()
    num_stats["Kurt"] = data[num_cols].kurt()
    cat_stats = data[cat_cols].describe().T
    return num_cols, cat_cols, num_stats, cat_stats


In [ ]:
def plot_distribucion_variables_numericas(data, num_cols):
    n_cols = 4
    n_rows = math.ceil(len(num_cols) / n_cols)
    fig, axes = plt.subplots(
        nrows=n_rows * 2,
        ncols=n_cols,
        figsize=(14, 6 * n_rows),
        sharex=False,
        gridspec_kw={
            "height_ratios": [3, 1] * n_rows
        },
    )
    for i, col in enumerate(num_cols):
        col_idx = i % n_cols
        row_idx = (i // n_cols) * 2
        ax_curve = axes[row_idx, col_idx]
        ax_box = axes[row_idx + 1, col_idx]
        ax_box.sharex(ax_curve)
        kde = col != "morning_alarm_snoozes"
        sns.histplot(
            data=data, x=col, kde=kde, ax=ax_curve, color="steelblue", alpha=0.4
        )
        ax_curve.set_title(col, fontsize=14, pad=10)
        ax_curve.set_xlabel("")
        ax_curve.set_ylabel("Frecuencia/Densidad")
        ax_curve.tick_params(
            labelbottom=False
        )
        ax_curve.grid(True, linestyle="--", alpha=0.5)
        sns.boxplot(data=data, x=col, ax=ax_box, color="lightblue", width=0.4)
        ax_box.set_xlabel("Valores")
        ax_box.grid(True, linestyle="--", alpha=0.5)
        mediana = data[col].median()
        media = data[col].mean()
        for ax in [ax_curve, ax_box]:
            ax.axvline(media, color='green', linestyle='--', alpha=0.8)
    for i in range(len(num_cols), n_rows * n_cols):
        col_idx = i % n_cols
        row_idx = (i // n_cols) * 2
        fig.delaxes(axes[row_idx, col_idx])
        fig.delaxes(axes[row_idx + 1, col_idx])
    plt.suptitle(
        "Distribución y Outliers de Variables Numéricas", fontsize=16, y=1
    )
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_distribucion_variables_categoricas(data, cat_cols):
    n_cols = 3
    n_rows = math.ceil(len(cat_cols) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 5 * n_rows))
    axes = axes.flatten()
    for i, col in enumerate(cat_cols):
        if col == "sleep_debt_category":
            freq_rel = (
                data[col]
                .value_counts(normalize=True, sort=False)
                * 100
            )
        else:
            freq_rel = data[col].value_counts(normalize=True) * 100
        bars = axes[i].bar(freq_rel.index, freq_rel.values, color="steelblue")
        axes[i].bar_label(bars, fmt="%.1f%%", padding=-12, fontsize=10)
        axes[i].set_title(col, fontsize=14)
        axes[i].set_ylabel("Porcentaje (%)")
        if col == "blue_light_filter_active":
            axes[i].set_xticks([0, 1])
            axes[i].set_xticklabels(['0', '1'])
        axes[i].tick_params(axis='x', rotation=70)
    for i in range(len(cat_cols), len(axes)):
        fig.delaxes(axes[i])
    plt.suptitle("Distribución de variables categóricas", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_analisis_bivariado(data):
    fig = plt.figure(figsize=(10, 6))
    sns.boxplot(data=data, x='sleep_debt_category', y='total_sleep_hours', hue="sleep_debt_category")
    plt.xticks(rotation=20)
    plt.title("Horas totales de sueño por categoría de deuda")
    plt.show()

    fig = plt.figure(figsize=(10, 6))
    sns.boxplot(data=data, x='sleep_debt_category', y='morning_alarm_snoozes', hue="sleep_debt_category")
    plt.xticks(rotation=20)
    plt.title("Aplazamientos de la alarma por categoría de deuda")
    plt.show()

    fig = plt.figure(figsize=(10, 6))
    sns.boxplot(data=data, x="sleep_debt_category", y="bedtime_phone_minutes", hue="sleep_debt_category")
    plt.xticks(rotation=20)
    plt.title("Uso del teléfono antes de dormir por categoría de deuda")
    plt.show()

    fig = plt.figure(figsize=(10, 6))
    sns.boxplot(data=data, x="sleep_debt_category", y="sleep_latency_min", hue="sleep_debt_category")
    plt.xticks(rotation=20)
    plt.title("Latencia del sueño por categoría de deuda")
    plt.show()

    tabla = pd.crosstab(data["chronotype"], data["sleep_debt_category"], normalize="index") * 100
    tabla.plot(kind="bar", stacked=True, figsize=(10, 6))
    plt.ylabel("Porcentaje (%)")
    plt.xlabel("")
    plt.title("Categoría de deuda de sueño según cronotipo")
    plt.xticks(rotation=20)
    plt.legend(title="Categoría de deuda de sueño")
    plt.show()

    fig = plt.figure(figsize=(10, 6))
    sns.regplot(data=data, x='bedtime_phone_minutes', y='sleep_latency_min', scatter_kws={'alpha': 0.5}, line_kws={'color': 'red'})
    plt.title("Relación entre uso del teléfono y latencia del sueño")
    plt.show()

    fig = plt.figure(figsize=(10, 6))
    sns.scatterplot(data=data, x='bedtime_phone_minutes', y='total_sleep_hours', hue='sleep_debt_category', alpha=0.7)
    plt.title("Relación entre uso del teléfono y horas totales de sueño")
    plt.show()

    fig = plt.figure(figsize=(10, 6))
    sns.boxplot(data=data, x='blue_light_filter_active', y='total_sleep_hours')
    plt.title("Horas totales de sueño según uso del filtro de luz azul")
    plt.show()

    fig = plt.figure(figsize=(10, 6))
    sns.boxplot(data=data, x='occupation_type', y='total_sleep_hours')
    plt.xticks(rotation=25)
    plt.title("Horas totales de sueño según tipo de ocupación")
    plt.show()

    fig = plt.figure(figsize=(10, 7))
    sns.boxplot(data=data, x='chronotype', y='total_sleep_hours')
    plt.xticks(rotation=30)
    plt.title("Horas totales de sueño según cronotipo")
    plt.show()

In [ ]:
def plot_analisis_multivariado(data, num_cols):
    fig = plt.figure(figsize=(14, 10))
    corr = data.select_dtypes(include='number').corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title("Mapa de correlación entre variables numéricas")
    plt.xticks(rotation=80)
    plt.show()

    sleep_debt_ordinal = data["sleep_debt_category"].cat.codes
    corr_target = data[num_cols].corrwith(sleep_debt_ordinal, method="spearman").sort_values()
    plt.figure(figsize=(8, 6))
    sns.barplot(x=corr_target.values, y=corr_target.index)
    plt.xlabel("Correlación de Spearman")
    plt.ylabel("")
    plt.title("Relación con Sleep Debt Category")
    plt.grid(True, linestyle="--", alpha=0.5, axis="x")
    plt.show()

In [ ]:
def split_data(data, target, val_size=0.15, test_size=0.15, random_state=777):
    X = data.drop(columns=[target])
    y = data[target].astype(str)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=val_size + test_size,
        random_state=random_state,
        stratify=y
    )
    relative_test_size = test_size / (val_size + test_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=relative_test_size,
        random_state=random_state,
        stratify=y_temp
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


In [ ]:
def evaluar_modelo(y_real, y_pred):
    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "recall_macro": recall_score(
            y_real,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "f1_macro": f1_score(
            y_real,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_real,
            y_pred
        ),
        "gmean": geometric_mean_score(
            y_real,
            y_pred,
            average="multiclass"
        )
    }

In [ ]:
def registrar_matriz_confusion(
    y_real,
    y_pred,
    titulo,
    artifact_file,
    mostrar=True
):

    clases_orden = [
    "Optimal Recovery",
    "Mild Deficit",
    "Moderate Debt",
    "Severe Sleep Debt"
    ]
    
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_real,
        y_pred,
        labels=clases_orden,
        cmap="Blues",
        xticks_rotation=30,
        ax=ax
    )
    ax.set_title(titulo)
    plt.tight_layout()
    mlflow.log_figure(
        fig,
        artifact_file
    )
    if mostrar:
        plt.show()
    plt.close(fig)

In [ ]:
def create_preprocessor(data, target):
    numeric_features = data.select_dtypes(
        include=["number"]
    ).columns.tolist()
    categorical_features = data.select_dtypes(
        include=["str", "category"]
    ).columns.tolist()
    categorical_features.remove(target)
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features)
        ]
    )
    return preprocessor

def train_model(
    X_train,
    y_train,
    preprocessor,
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=777
):
    model = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            (
                "classifier",
                LogisticRegression(
                    C=C,
                    solver=solver,
                    max_iter=max_iter,
                    random_state=random_state
                )
            )
        ]
    )
    model.fit(X_train, y_train)
    return model


In [ ]:
def configure_mlflow(
    tracking_uri="http://127.0.0.1:5000",
    experiment_name="Logistic Regression Sleep Debt"
):
    mlflow.set_tracking_uri(uri=tracking_uri)
    mlflow.set_experiment(experiment_name)


In [ ]:
def evaluate_model(model, X_data, y_data, split_name):
    y_pred = model.predict(X_data)
    metrics = evaluar_modelo(y_data, y_pred)
    prefixed_metrics = {
        f"{split_name}_{metric_name}": metric_value
        for metric_name, metric_value in metrics.items()
    }
    mlflow.log_metrics(prefixed_metrics)
    results = pd.DataFrame({
        "Valor_Real": y_data,
        "Prediccion": y_pred
    })
    file_name = f"predictions_{split_name}.csv"
    results.to_csv(file_name, index=False)
    mlflow.log_artifact(file_name, artifact_path="predictions")
    registrar_matriz_confusion(
        y_data,
        y_pred,
        f"Matriz de confusión - {split_name}",
        f"confusion_matrix/{split_name}.png",
        mostrar=True
    )
    return metrics


In [ ]:
def run_experiment(
    data,
    target,
    val_size=0.15,
    test_size=0.15,
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=777
):
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(
        data,
        target,
        val_size=val_size,
        test_size=test_size,
        random_state=random_state
    )
    preprocessor = create_preprocessor(data, target)
    with mlflow.start_run(run_name="LogisticRegression - train validation test"):
        mlflow.log_params({
            "C": C,
            "solver": solver,
            "max_iter": max_iter,
            "random_state": random_state,
            "validation_size": val_size,
            "test_size": test_size
        })
        validation_model = train_model(
            X_train,
            y_train,
            preprocessor,
            C=C,
            solver=solver,
            max_iter=max_iter,
            random_state=random_state
        )
        train_metrics = evaluate_model(
            validation_model,
            X_train,
            y_train,
            "train"
        )
        validation_metrics = evaluate_model(
            validation_model,
            X_val,
            y_val,
            "validation"
        )
        X_train_final = pd.concat([X_train, X_val])
        y_train_final = pd.concat([y_train, y_val])
        final_model = train_model(
            X_train_final,
            y_train_final,
            preprocessor,
            C=C,
            solver=solver,
            max_iter=max_iter,
            random_state=random_state
        )
        test_metrics = evaluate_model(
            final_model,
            X_test,
            y_test,
            "test"
        )
        signature = infer_signature(
            X_train_final,
            final_model.predict(X_train_final)
        )
        mlflow.sklearn.log_model(
            sk_model=final_model,
            name="model",
            signature=signature,
            input_example=X_train_final.head(5)
        )
    return final_model, {
        "train": train_metrics,
        "validation": validation_metrics,
        "test": test_metrics
    }


In [ ]:
def main(filepath):
    data = load_data(filepath)
    explore_data(data)
    data = clean_data(data)
    num_cols, cat_cols, num_stats, cat_stats = explore_variables(data)
    plot_distribucion_variables_numericas(data, num_cols)
    plot_distribucion_variables_categoricas(data, cat_cols)
    plot_analisis_bivariado(data)
    plot_analisis_multivariado(data, num_cols)
    configure_mlflow()
    run_experiment(data, "sleep_debt_category")


In [ ]:
main(filepath="../data/raw/bedtime_screentime_sleep_debt.csv")
